# Day 20 - Silver Cleansing Pattern

**Topic:** Silver Cleansing Pattern  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึกเปลี่ยน Bronze table ที่ยังเป็น raw/traceable data ให้เป็น Silver table ที่ standardized, typed, deduplicated และมี technical DQ flags

วันนี้เราจะต่อจากแนวคิด Bronze ingestion โดยจำลอง Bronze table ขนาดเล็ก แล้วใช้ PySpark ทำ Silver cleansing pattern แบบ production-aware: ทำความสะอาด string, cast data type, normalize status, deduplicate ด้วย business key, เก็บ audit columns สำคัญ และ write ผลลัพธ์เป็น Lakehouse table


## Goal of the day

1. อธิบายได้ว่า **Bronze** กับ **Silver** มีหน้าที่ต่างกันอย่างไรใน Lakehouse pipeline
2. อ่าน Bronze table แล้วแปลงเป็น Silver DataFrame ที่ field name, data type และ value format ชัดเจนขึ้น
3. ใช้ PySpark ทำ **standardization**, **type casting**, **technical DQ flags** และ **deduplication** ได้
4. เข้าใจว่าทำไม Silver ไม่ควร drop bad records แบบเงียบ ๆ โดยไม่มี traceability
5. เขียน Silver output เป็น Fabric Lakehouse table และตรวจผลหลัง write ได้


## Why it matters in production

ใน production pipeline, Bronze มักเก็บข้อมูลใกล้เคียง source เพื่อรักษา traceability ส่วน Silver คือจุดที่ data เริ่มถูกทำให้ใช้งานต่อได้จริงมากขึ้น เช่น schema ชัดเจน, date/number ถูก cast, status ถูก normalize และ duplicate ถูกจัดการด้วย business rule

Silver cleansing สำคัญเพราะ:

- downstream analytics ไม่ควรต้อง parse raw string เองทุกครั้ง
- field ที่เป็น date, timestamp, amount ควรถูก cast เป็น type ที่ถูกต้องก่อนใช้ต่อ
- duplicate records ต้องถูกจัดการด้วย business key + ordering rule ไม่ใช่ลบแบบสุ่ม
- records ที่มีปัญหาควรถูก flag ให้มองเห็นได้ ก่อนจะไปลงรายละเอียด DQ/quarantine ในวันถัดไป
- audit columns จาก Bronze ยังมีประโยชน์สำหรับ trace back ไปหา source file, batch และ run

Production takeaway:

> Silver layer คือจุดที่ raw data เริ่มกลายเป็น reliable dataset แต่ยังต้องรักษาร่องรอยกลับไปที่ Bronze/source ได้เสมอ


## Key concepts

| Concept | Meaning |
|---|---|
| Bronze table | ตารางที่เก็บ raw/ingested data พร้อม audit columns เพื่อ trace source ได้ |
| Silver table | ตารางที่ผ่าน cleansing, standardization, type casting และ deduplication แล้ว |
| Standardization | การทำ format ให้สม่ำเสมอ เช่น trim, lower, normalize status |
| Type casting | การเปลี่ยน raw string ให้เป็น data type ที่ใช้งานได้ เช่น date, timestamp, decimal |
| Business key | key ที่ใช้ระบุ entity/transaction ตาม business logic เช่น `order_id` |
| Deduplication rule | กติกาเลือก record ที่ควรเก็บ เช่น latest `source_updated_at` ต่อ `order_id` |
| Technical DQ | การ flag ปัญหาเชิงเทคนิคเบื้องต้น เช่น missing key, invalid amount, invalid date |
| Traceability | ความสามารถในการย้อนกลับไปหา source file, batch id, run id และ ingestion timestamp |


## Concept explanation

### Bronze กับ Silver ต่างกันอย่างไร?

Bronze ไม่ได้เน้นให้ข้อมูลสวยที่สุด แต่เน้นให้ตรวจสอบย้อนหลังได้:

- source ส่งอะไรมา
- มาจาก file หรือ batch ไหน
- ingest เข้ามาเมื่อไร
- run ไหนเป็นคนสร้างข้อมูลชุดนี้

Silver คือ layer ที่เริ่มทำให้ data ใช้ต่อได้จริง:

- column names ชัดเจนขึ้น
- string values ถูก trim / normalize
- date และ amount ถูก cast เป็น type ที่ถูกต้อง
- duplicate ถูกจัดการด้วย business rule
- records ที่มี technical issue ถูก flag ให้เห็น

พูดง่าย ๆ:

> Bronze = preserve raw and trace source  
> Silver = standardize, type, deduplicate, and make issues visible

### Silver cleansing pattern ควรทำอะไรบ้าง?

สำหรับ notebook วันนี้ เราจะใช้ flow แบบนี้:

1. Read Bronze table
2. Standardize raw fields
3. Cast field types
4. Normalize business values เช่น payment status
5. Create technical issue flags
6. Deduplicate by business key
7. Write Silver table
8. Validate output

### Technical DQ กับ Data Quality เต็มรูปแบบต่างกันอย่างไร?

วันนี้จะยังไม่ลงลึก rule engine, DQ log หรือ quarantine table เพราะ Day 21 และ Day 22 จะทำเรื่องนั้นต่อ

ใน Day 20 เราจะทำแค่ technical DQ flags เพื่อให้เห็นว่า record ไหนมีปัญหาเบื้องต้น เช่น:

- missing `order_id` หรือ `customer_id`
- amount เป็น null หรือไม่มากกว่า 0
- parse date ไม่ได้
- payment status ไม่รู้จัก

แนวคิดสำคัญคือ:

> Silver cleansing ไม่ควรทำให้ data issue หายเงียบ ๆ แต่ควรทำให้ issue เห็นง่ายขึ้นและพร้อมต่อยอดไป DQ/quarantine


## Mermaid diagram: Bronze to Silver Cleansing Flow

```mermaid
flowchart LR
    A[Bronze Table<br/>Raw + Audit Columns] --> B[Read Bronze]
    B --> C[Standardize Strings<br/>trim / lower / upper]
    C --> D[Cast Types<br/>date / timestamp / decimal]
    D --> E[Technical DQ Flags<br/>missing key / invalid amount / invalid date]
    E --> F[Deduplicate<br/>business key + latest timestamp]
    F --> G[Silver Table<br/>Typed + Cleaned + Traceable]
    G --> H[Downstream DQ / Quarantine / Gold]
```

Key idea:

> Silver ไม่ใช่แค่ clean data แต่เป็น layer ที่ทำให้ข้อมูลพร้อมตรวจสอบและพร้อมใช้ต่อมากขึ้น


## Setup / imports


In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 3, Finished, Available, Finished, False)

## Create mock data

ข้อมูลจำลองตั้งใจให้มีปัญหาที่เจอบ่อยใน source จริง เช่น:

- whitespace และ case ไม่สม่ำเสมอ
- amount เป็น string และมี comma/currency text
- date หลาย format
- duplicate `order_id`
- missing `customer_id`
- payment status หลายรูปแบบ
- audit columns จาก Bronze เช่น `source_file`, `batch_id`, `run_id`, `ingestion_timestamp_raw`


In [2]:
bronze_table_name = "day20_bronze_orders_raw"
silver_table_name = "day20_silver_orders"

bronze_orders_data = [
    (" o1001 ", " c001 ", " Alice ", " ALICE@Example.COM ", "+66 81-111-2222", "2026-05-01", "1,200.50", "SUCCESS", "pos", "orders_20260501_001.csv", "batch_001", "run_001", "2026-05-01 08:30:00", "2026-05-01 09:00:00"),
    ("O1002", "C002", "BOB", "bob@example.com", "081 222 3333", "01/05/2026", "850.00", "paid", "pos", "orders_20260501_001.csv", "batch_001", "run_001", "2026-05-01 08:35:00", "2026-05-01 09:00:00"),
    ("O1002", "C002", "Bob", " bob@example.com ", "081-222-3333", "2026-05-01", "900.00", "completed", "pos", "orders_20260501_002.csv", "batch_001", "run_001", "2026-05-01 10:00:00", "2026-05-01 10:10:00"),
    ("O1003", "C003", "Charlie", "CHARLIE@EXAMPLE.COM", None, "2026/05/02", "0", "success", "web", "orders_20260502_001.csv", "batch_002", "run_002", "2026-05-02 11:00:00", "2026-05-02 11:10:00"),
    ("O1004", "", "Diana", "diana@example.com", "0824445555", "2026-05-02", "THB 1,500.00", "waiting", "web", "orders_20260502_001.csv", "batch_002", "run_002", "2026-05-02 11:05:00", "2026-05-02 11:10:00"),
    ("O1005", "C005", "Ethan", None, "", "bad-date", "700.00", "error", "mobile", "orders_20260503_001.csv", "batch_003", "run_003", "2026-05-03 07:45:00", "2026-05-03 08:00:00"),
    ("O1006", "C006", "Fah", "fah@example.com", "0836667777", "2026-05-03", "-100.00", "success", "mobile", "orders_20260503_001.csv", "batch_003", "run_003", "2026-05-03 07:50:00", "2026-05-03 08:00:00"),
    ("O1007", "C007", "Grace", "N/A", "0848889999", "2026-05-04", "ABC", "unknown_status", "partner", "orders_20260504_001.csv", "batch_004", "run_004", "2026-05-04 08:10:00", "2026-05-04 08:20:00"),
]

bronze_orders_schema = T.StructType([
    T.StructField("order_id", T.StringType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("customer_email", T.StringType(), True),
    T.StructField("customer_phone", T.StringType(), True),
    T.StructField("order_date_raw", T.StringType(), True),
    T.StructField("order_amount_raw", T.StringType(), True),
    T.StructField("payment_status_raw", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("source_file", T.StringType(), True),
    T.StructField("batch_id", T.StringType(), True),
    T.StructField("run_id", T.StringType(), True),
    T.StructField("source_updated_at_raw", T.StringType(), True),
    T.StructField("ingestion_timestamp_raw", T.StringType(), True),
])

df_bronze_orders_raw = spark.createDataFrame(bronze_orders_data, bronze_orders_schema)

df_bronze_orders_raw.show(truncate=False)
df_bronze_orders_raw.printSchema()
print("Bronze source record count:", df_bronze_orders_raw.count())

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 4, Finished, Available, Finished, False)

+--------+-----------+-------------+-------------------+---------------+--------------+----------------+------------------+-------------+-----------------------+---------+-------+---------------------+-----------------------+
|order_id|customer_id|customer_name|customer_email     |customer_phone |order_date_raw|order_amount_raw|payment_status_raw|source_system|source_file            |batch_id |run_id |source_updated_at_raw|ingestion_timestamp_raw|
+--------+-----------+-------------+-------------------+---------------+--------------+----------------+------------------+-------------+-----------------------+---------+-------+---------------------+-----------------------+
| o1001  | c001      | Alice       | ALICE@Example.COM |+66 81-111-2222|2026-05-01    |1,200.50        |SUCCESS           |pos          |orders_20260501_001.csv|batch_001|run_001|2026-05-01 08:30:00  |2026-05-01 09:00:00    |
|O1002   |C002       |BOB          |bob@example.com    |081 222 3333   |01/05/2026    |850.00   

### Write mock Bronze table

In [3]:
(
    df_bronze_orders_raw
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table_name)
)

spark.read.table(bronze_table_name).show(truncate=False)

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 5, Finished, Available, Finished, False)

+--------+-----------+-------------+-------------------+---------------+--------------+----------------+------------------+-------------+-----------------------+---------+-------+---------------------+-----------------------+
|order_id|customer_id|customer_name|customer_email     |customer_phone |order_date_raw|order_amount_raw|payment_status_raw|source_system|source_file            |batch_id |run_id |source_updated_at_raw|ingestion_timestamp_raw|
+--------+-----------+-------------+-------------------+---------------+--------------+----------------+------------------+-------------+-----------------------+---------+-------+---------------------+-----------------------+
| o1001  | c001      | Alice       | ALICE@Example.COM |+66 81-111-2222|2026-05-01    |1,200.50        |SUCCESS           |pos          |orders_20260501_001.csv|batch_001|run_001|2026-05-01 08:30:00  |2026-05-01 09:00:00    |
|O1002   |C002       |Bob          | bob@example.com   |081-222-3333   |2026-05-01    |900.00   

## PySpark code examples

ใน section นี้จะไล่จาก Bronze → Silver แบบ step-by-step โดยตั้งใจให้เห็น intermediate DataFrame เพื่อ debug ได้ง่าย


### Example 1: Read Bronze table and inspect raw data

ก่อน clean ต้องอ่านข้อมูลจาก Bronze และดู schema/raw values ก่อนเสมอ เพื่อให้รู้ว่าตอนนี้ field ไหนยังเป็น string, field ไหนมี whitespace, field ไหนมี format ไม่สม่ำเสมอ


In [4]:
df_bronze = spark.read.table(bronze_table_name)

df_bronze.select(
    "order_id",
    "customer_id",
    "order_date_raw",
    "order_amount_raw",
    "payment_status_raw",
    "source_file",
    "batch_id",
    "run_id",
).show(truncate=False)

df_bronze.printSchema()
print("Bronze table record count:", df_bronze.count())

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 6, Finished, Available, Finished, False)

+--------+-----------+--------------+----------------+------------------+-----------------------+---------+-------+
|order_id|customer_id|order_date_raw|order_amount_raw|payment_status_raw|source_file            |batch_id |run_id |
+--------+-----------+--------------+----------------+------------------+-----------------------+---------+-------+
| o1001  | c001      |2026-05-01    |1,200.50        |SUCCESS           |orders_20260501_001.csv|batch_001|run_001|
|O1002   |C002       |2026-05-01    |900.00          |completed         |orders_20260501_002.csv|batch_001|run_001|
|O1004   |           |2026-05-02    |THB 1,500.00    |waiting           |orders_20260502_001.csv|batch_002|run_002|
|O1006   |C006       |2026-05-03    |-100.00         |success           |orders_20260503_001.csv|batch_003|run_003|
|O1002   |C002       |01/05/2026    |850.00          |paid              |orders_20260501_001.csv|batch_001|run_001|
|O1007   |C007       |2026-05-04    |ABC             |unknown_status    

### Example 2: Standardize raw string fields

ขั้นแรกของ Silver คือทำให้ string fields มี format ที่คาดเดาได้มากขึ้น เช่น trim whitespace, normalize case และเปลี่ยนค่าว่าง หรือ `N/A` ให้เป็น null


In [5]:
def null_if_blank(column_name: str):
    trimmed_col = F.trim(F.col(column_name))
    lowered_col = F.lower(trimmed_col)

    return (
        F.when(
            F.col(column_name).isNull()
            | (trimmed_col == "")
            | lowered_col.isin("null", "n/a", "na", "none"),
            F.lit(None),
        )
        .otherwise(trimmed_col)
    )


df_standardized = df_bronze.select(
    F.upper(null_if_blank("order_id")).alias("order_id"),
    F.upper(null_if_blank("customer_id")).alias("customer_id"),
    F.initcap(F.lower(null_if_blank("customer_name"))).alias("customer_name"),
    F.lower(null_if_blank("customer_email")).alias("customer_email"),
    null_if_blank("customer_phone").alias("customer_phone_raw"),
    null_if_blank("order_date_raw").alias("order_date_raw"),
    null_if_blank("order_amount_raw").alias("order_amount_raw"),
    F.lower(null_if_blank("payment_status_raw")).alias("payment_status_raw"),
    F.lower(null_if_blank("source_system")).alias("source_system"),
    null_if_blank("source_file").alias("source_file"),
    null_if_blank("batch_id").alias("batch_id"),
    null_if_blank("run_id").alias("run_id"),
    null_if_blank("source_updated_at_raw").alias("source_updated_at_raw"),
    null_if_blank("ingestion_timestamp_raw").alias("ingestion_timestamp_raw"),
)

df_standardized.show(truncate=False)

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 7, Finished, Available, Finished, False)

+--------+-----------+-------------+-------------------+------------------+--------------+----------------+------------------+-------------+-----------------------+---------+-------+---------------------+-----------------------+
|order_id|customer_id|customer_name|customer_email     |customer_phone_raw|order_date_raw|order_amount_raw|payment_status_raw|source_system|source_file            |batch_id |run_id |source_updated_at_raw|ingestion_timestamp_raw|
+--------+-----------+-------------+-------------------+------------------+--------------+----------------+------------------+-------------+-----------------------+---------+-------+---------------------+-----------------------+
|O1001   |C001       |Alice        |alice@example.com  |+66 81-111-2222   |2026-05-01    |1,200.50        |success           |pos          |orders_20260501_001.csv|batch_001|run_001|2026-05-01 08:30:00  |2026-05-01 09:00:00    |
|O1002   |C002       |Bob          |bob@example.com    |081-222-3333      |2026-05-0

### Example 3: Cast dates, timestamps, phone, amount, and status

ขั้นนี้เปลี่ยน raw text ให้เป็น type ที่ใช้ต่อได้จริง:

- `order_date` เป็น DateType
- `source_updated_at` และ `ingestion_timestamp` เป็น TimestampType
- `order_amount` เป็น DecimalType
- `payment_status` ถูก normalize เป็นกลุ่มมาตรฐาน เช่น `success`, `failed`, `pending`, `unknown`


In [6]:
amount_text = F.regexp_replace(F.col("order_amount_raw"), "[^0-9.-]", "")  # Remove anything that is not a digit, dot, or minus sign.
phone_digits = F.regexp_replace(F.col("customer_phone_raw"), "[^0-9]", "")  # Remove anything that is not a digit.
status_col = F.col("payment_status_raw")

df_typed = (
    df_standardized
    .withColumn("customer_phone_digits", F.when(F.length(phone_digits) == 0, F.lit(None)).otherwise(phone_digits))
    .withColumn("order_amount_text", amount_text)
    .withColumn(
        "order_amount",
        F.when(
            F.col("order_amount_text").isNull() | (F.col("order_amount_text") == ""),
            F.lit(None).cast("decimal(12,2)"),
        ).otherwise(F.col("order_amount_text").cast("decimal(12,2)")),
    )
    .withColumn(
        "order_date",
        F.coalesce(
            F.to_date("order_date_raw", "yyyy-MM-dd"),  # Convert string to DateType using the expected input format.
            F.to_date("order_date_raw", "dd/MM/yyyy"),
            F.to_date("order_date_raw", "yyyy/MM/dd"),
        ),
    )
    .withColumn("source_updated_at", F.to_timestamp("source_updated_at_raw", "yyyy-MM-dd HH:mm:ss"))  # Convert string to TimestampType using the expected input format.
    .withColumn("ingestion_timestamp", F.to_timestamp("ingestion_timestamp_raw", "yyyy-MM-dd HH:mm:ss"))
    .withColumn(
        "payment_status",
        F.when(status_col.isin("success", "paid", "completed"), F.lit("success"))
        .when(status_col.isin("failed", "error", "rejected"), F.lit("failed"))
        .when(status_col.isin("pending", "waiting"), F.lit("pending"))
        .otherwise(F.lit("unknown")),
    )
    .drop("order_amount_text")
)

df_typed.select(
    "order_id",
    "customer_id",
    "customer_email",
    "customer_phone_digits",
    "order_date",
    "order_amount",
    "payment_status",
    "source_updated_at",
    "ingestion_timestamp",
).show(truncate=False)

df_typed.printSchema()

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 8, Finished, Available, Finished, False)

+--------+-----------+-------------------+---------------------+----------+------------+--------------+-------------------+-------------------+
|order_id|customer_id|customer_email     |customer_phone_digits|order_date|order_amount|payment_status|source_updated_at  |ingestion_timestamp|
+--------+-----------+-------------------+---------------------+----------+------------+--------------+-------------------+-------------------+
|O1001   |C001       |alice@example.com  |66811112222          |2026-05-01|1200.50     |success       |2026-05-01 08:30:00|2026-05-01 09:00:00|
|O1002   |C002       |bob@example.com    |0812223333           |2026-05-01|900.00      |success       |2026-05-01 10:00:00|2026-05-01 10:10:00|
|O1004   |NULL       |diana@example.com  |0824445555           |2026-05-02|1500.00     |pending       |2026-05-02 11:05:00|2026-05-02 11:10:00|
|O1006   |C006       |fah@example.com    |0836667777           |2026-05-03|-100.00     |success       |2026-05-03 07:50:00|2026-05-03 08

### Example 4: Create technical DQ flags

ก่อน deduplicate หรือ write Silver ควรทำให้ปัญหาสำคัญมองเห็นได้ก่อน เช่น missing business key, invalid amount, invalid date หรือ unknown status

วันนี้ยังไม่แยก quarantine table เพราะจะเป็นหัวข้อของ Day 22 แต่เราจะใส่ `technical_issue_reason` และ `is_technical_valid` ไว้ใน Silver table เพื่อไม่ให้ issue หายเงียบ


In [7]:
df_flagged = (
    df_typed
    .withColumn(
        "technical_issue_reason",
        F.concat_ws(
            "|",
            F.when(
                F.col("order_id").isNull() | F.col("customer_id").isNull(),
                F.lit("missing_business_key"),
            ),
            F.when(
                F.col("order_amount").isNull() | (F.col("order_amount") <= 0),
                F.lit("invalid_amount"),
            ),
            F.when(F.col("order_date").isNull(), F.lit("invalid_order_date")),
            F.when(F.col("payment_status") == "unknown", F.lit("unknown_payment_status")),
            F.when(
                F.col("customer_email").isNull() & F.col("customer_phone_digits").isNull(),
                F.lit("missing_contact"),
            ),
        ),
    )
    .withColumn("is_technical_valid", F.col("technical_issue_reason") == "")
)

df_flagged.select(
    "order_id",
    "customer_id",
    "order_date",
    "order_amount",
    "payment_status",
    "technical_issue_reason",
    "is_technical_valid",
).show(truncate=False)

df_flagged.groupBy("is_technical_valid").count().orderBy("is_technical_valid").show()

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 9, Finished, Available, Finished, False)

+--------+-----------+----------+------------+--------------+-------------------------------------+------------------+
|order_id|customer_id|order_date|order_amount|payment_status|technical_issue_reason               |is_technical_valid|
+--------+-----------+----------+------------+--------------+-------------------------------------+------------------+
|O1001   |C001       |2026-05-01|1200.50     |success       |                                     |true              |
|O1002   |C002       |2026-05-01|900.00      |success       |                                     |true              |
|O1004   |NULL       |2026-05-02|1500.00     |pending       |missing_business_key                 |false             |
|O1006   |C006       |2026-05-03|-100.00     |success       |invalid_amount                       |false             |
|O1002   |C002       |2026-05-01|850.00      |success       |                                     |true              |
|O1007   |C007       |2026-05-04|NULL        |un

### Example 5: Check duplicate candidates before deduplication

ก่อน deduplicate ควรตรวจว่า key ไหนซ้ำ และซ้ำกี่ record เพื่อไม่ให้เราเลือก record ทิ้งโดยไม่รู้ตัว

ในตัวอย่างนี้ `O1002` มี 2 records และเราจะเก็บ record ที่มี `source_updated_at` ล่าสุด


In [8]:
df_duplicate_candidates = (
    df_flagged
    .filter(F.col("order_id").isNotNull())
    .groupBy("order_id")
    .agg(
        F.count("*").alias("record_count"),
        F.max("source_updated_at").alias("latest_source_updated_at"),
        F.collect_set("source_file").alias("source_files"),  # Collect unique values into an array.
    )
    .filter(F.col("record_count") > 1)
    .orderBy("order_id")
)

df_duplicate_candidates.show(truncate=False)

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 10, Finished, Available, Finished, False)

+--------+------------+------------------------+--------------------------------------------------+
|order_id|record_count|latest_source_updated_at|source_files                                      |
+--------+------------+------------------------+--------------------------------------------------+
|O1002   |2           |2026-05-01 10:00:00     |[orders_20260501_002.csv, orders_20260501_001.csv]|
+--------+------------+------------------------+--------------------------------------------------+



### Example 6: Deduplicate by business key and latest timestamp

Deduplication ที่ดีควรมี ordering rule ชัดเจน ไม่ควรใช้ `dropDuplicates()` อย่างเดียวกับข้อมูลที่มีหลาย version เพราะ Spark จะไม่ได้รู้ว่า record ไหนคือ record ที่ถูกต้องที่สุดในเชิง business

Rule วันนี้:

- business key = `order_id`
- ถ้า `order_id` ซ้ำ ให้เก็บ record ที่ `source_updated_at` ล่าสุด
- ถ้า timestamp เท่ากัน ให้ใช้ `ingestion_timestamp` เป็น tie-breaker
- records ที่ `order_id` เป็น null จะไม่ถูกจับรวมเป็นกลุ่มเดียวกันแบบผิด ๆ


In [10]:
df_for_dedup = df_flagged.withColumn(
    "dedup_key",
    F.when(
        F.col("order_id").isNull(),
        F.concat(F.lit("__missing_order_id__"), F.monotonically_increasing_id().cast("string")),  # Create a placeholder key using a unique non-consecutive row id.
    ).otherwise(F.col("order_id")),
)

silver_dedup_window = (
    Window
    .partitionBy("dedup_key")
    .orderBy(
        F.col("source_updated_at").desc_nulls_last(),  # Sort values in descending order and place null values at the end.
        F.col("ingestion_timestamp").desc_nulls_last(),
        F.col("source_file").desc_nulls_last(),
    )
)

df_ranked = df_for_dedup.withColumn("record_rank", F.row_number().over(silver_dedup_window))

df_silver_all = (
    df_ranked
    .filter(F.col("record_rank") == 1)
    .drop("dedup_key", "record_rank")
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "order_id",
        "customer_id",
        "customer_name",
        "customer_email",
        "customer_phone_digits",
        "order_date",
        "order_amount",
        "payment_status",
        "source_system",
        "source_file",
        "batch_id",
        "run_id",
        "source_updated_at",
        "ingestion_timestamp",
        "technical_issue_reason",
        "is_technical_valid",
        "silver_processed_at",
    )
)

df_silver_all.orderBy("order_id").show(truncate=False)
print("Silver record count after dedup:", df_silver_all.count())

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 12, Finished, Available, Finished, False)

+--------+-----------+-------------+-------------------+---------------------+----------+------------+--------------+-------------+-----------------------+---------+-------+-------------------+-------------------+-------------------------------------+------------------+--------------------------+
|order_id|customer_id|customer_name|customer_email     |customer_phone_digits|order_date|order_amount|payment_status|source_system|source_file            |batch_id |run_id |source_updated_at  |ingestion_timestamp|technical_issue_reason               |is_technical_valid|silver_processed_at       |
+--------+-----------+-------------+-------------------+---------------------+----------+------------+--------------+-------------+-----------------------+---------+-------+-------------------+-------------------+-------------------------------------+------------------+--------------------------+
|O1001   |C001       |Alice        |alice@example.com  |66811112222          |2026-05-01|1200.50     |succ

### Example 7: Validate Silver output before write

หลัง transform เสร็จ ควร validate แบบง่าย ๆ ก่อน write เช่น:

- จำนวน record หลัง dedup
- schema ของ typed columns
- duplicate key ที่เหลือ
- summary ของ valid/invalid technical status


In [11]:
print("Bronze record count:", df_bronze.count())
print("Silver record count:", df_silver_all.count())

print("Silver schema:")
df_silver_all.printSchema()

print("Duplicate order_id check after dedup:")
(
    df_silver_all
    .filter(F.col("order_id").isNotNull())
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .show(truncate=False)
)

print("Technical validity summary:")
df_silver_all.groupBy("is_technical_valid").count().orderBy("is_technical_valid").show()

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 13, Finished, Available, Finished, False)

Bronze record count: 8
Silver record count: 7
Silver schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_phone_digits: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_amount: decimal(12,2) (nullable = true)
 |-- payment_status: string (nullable = false)
 |-- source_system: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- source_updated_at: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- technical_issue_reason: string (nullable = false)
 |-- is_technical_valid: boolean (nullable = false)
 |-- silver_processed_at: timestamp (nullable = false)

Duplicate order_id check after dedup:
+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+

Technical validity summary:

### Example 8: Write Silver table


In [12]:
(
    df_silver_all
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table_name)
)

df_silver_from_table = spark.read.table(silver_table_name)

df_silver_from_table.orderBy("order_id").show(truncate=False)
print("Silver table count:", df_silver_from_table.count())

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 14, Finished, Available, Finished, False)

+--------+-----------+-------------+-------------------+---------------------+----------+------------+--------------+-------------+-----------------------+---------+-------+-------------------+-------------------+-------------------------------------+------------------+-------------------------+
|order_id|customer_id|customer_name|customer_email     |customer_phone_digits|order_date|order_amount|payment_status|source_system|source_file            |batch_id |run_id |source_updated_at  |ingestion_timestamp|technical_issue_reason               |is_technical_valid|silver_processed_at      |
+--------+-----------+-------------+-------------------+---------------------+----------+------------+--------------+-------------+-----------------------+---------+-------+-------------------+-------------------+-------------------------------------+------------------+-------------------------+
|O1001   |C001       |Alice        |alice@example.com  |66811112222          |2026-05-01|1200.50     |success

### Example 9: Create a downstream-ready Silver view

บาง pipeline อาจให้ Silver table เก็บทั้ง valid และ invalid technical records พร้อม flag เพื่อ trace issue ได้ ส่วน downstream job จะ filter เฉพาะ records ที่พร้อมใช้ต่อ

ตัวอย่างนี้ยังไม่ write เป็น table แยก เพราะ Day 21–22 จะลงลึกเรื่อง DQ และ quarantine ต่อ


In [13]:
df_silver_ready_orders = (
    df_silver_from_table
    .filter(F.col("is_technical_valid"))  # Short form of: filter where is_technical_valid == True.
    .select(
        "order_id",
        "customer_id",
        "customer_name",
        "order_date",
        "order_amount",
        "payment_status",
        "source_system",
    )
    .orderBy("order_id")
)

df_silver_issue_records = (
    df_silver_from_table
    .filter(~F.col("is_technical_valid"))  # Short form of: filter where is_technical_valid == False.
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "order_amount",
        "payment_status",
        "technical_issue_reason",
        "source_file",
        "batch_id",
    )
    .orderBy("order_id")
)

print("Ready records:")
df_silver_ready_orders.show(truncate=False)

print("Records with technical issues:")
df_silver_issue_records.show(truncate=False)

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 15, Finished, Available, Finished, False)

Ready records:
+--------+-----------+-------------+----------+------------+--------------+-------------+
|order_id|customer_id|customer_name|order_date|order_amount|payment_status|source_system|
+--------+-----------+-------------+----------+------------+--------------+-------------+
|O1001   |C001       |Alice        |2026-05-01|1200.50     |success       |pos          |
|O1002   |C002       |Bob          |2026-05-01|900.00      |success       |pos          |
+--------+-----------+-------------+----------+------------+--------------+-------------+

Records with technical issues:
+--------+-----------+----------+------------+--------------+-------------------------------------+-----------------------+---------+
|order_id|customer_id|order_date|order_amount|payment_status|technical_issue_reason               |source_file            |batch_id |
+--------+-----------+----------+------------+--------------+-------------------------------------+-----------------------+---------+
|O1003   |C

## Quick recap

| Question | Answer |
|---|---|
| Silver layer ทำหน้าที่หลักอะไร? | เปลี่ยน Bronze raw data ให้เป็น standardized, typed, deduplicated และพร้อมใช้ต่อมากขึ้น |
| Bronze audit columns ยังจำเป็นใน Silver ไหม? | จำเป็นในหลายกรณี เพราะช่วย trace กลับไปหา source file, batch และ run ได้ |
| ควร cast amount/date ก่อน clean string ไหม? | ไม่ควร ควร clean raw string เช่น comma/currency/date format ก่อน cast |
| Dedup ด้วย `dropDuplicates()` อย่างเดียวพอไหม? | ไม่พอสำหรับ data ที่มีหลาย version ควรมี business key และ ordering rule ชัดเจน |
| Silver ควร drop bad records เงียบ ๆ ไหม? | ไม่ควร อย่างน้อยควร flag issue หรือเตรียมส่งต่อไป DQ/quarantine pattern |


## Exercises


### Exercise 1: Create contact completeness flag

สร้าง DataFrame ชื่อ `df_contact_check` จาก `df_silver_from_table` เพื่อตรวจว่าลูกค้าแต่ละ order มี contact อย่างน้อย 1 ช่องทางหรือไม่

Requirements:

- ใช้ columns: `order_id`, `customer_id`, `customer_email`, `customer_phone_digits`
- สร้าง column `has_contact`
- `has_contact = true` เมื่อมี email หรือ phone อย่างน้อย 1 ค่า
- แสดงผลด้วย `.show(truncate=False)`

Expected result:

- records ที่ไม่มีทั้ง email และ phone ควรได้ `has_contact = false`


In [14]:
df_contact_check = (
    df_silver_from_table
    .select(
        "order_id",
        "customer_id",
        "customer_email",
        "customer_phone_digits",
    )
    .withColumn(
        "has_contact",
        F.col("customer_email").isNotNull() | F.col("customer_phone_digits").isNotNull(),
    )
    .orderBy("order_id")
)

df_contact_check.show(truncate=False)

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 16, Finished, Available, Finished, False)

+--------+-----------+-------------------+---------------------+-----------+
|order_id|customer_id|customer_email     |customer_phone_digits|has_contact|
+--------+-----------+-------------------+---------------------+-----------+
|O1001   |C001       |alice@example.com  |66811112222          |true       |
|O1002   |C002       |bob@example.com    |0812223333           |true       |
|O1003   |C003       |charlie@example.com|NULL                 |true       |
|O1004   |NULL       |diana@example.com  |0824445555           |true       |
|O1005   |C005       |NULL               |NULL                 |false      |
|O1006   |C006       |fah@example.com    |0836667777           |true       |
|O1007   |C007       |NULL               |0848889999           |true       |
+--------+-----------+-------------------+---------------------+-----------+



### Exercise 2: Create ready-only DataFrame for downstream jobs

สร้าง DataFrame ชื่อ `df_downstream_ready_orders` จาก `df_silver_from_table` สำหรับส่งต่อไป downstream transformation แบบง่าย ๆ

Requirements:

- filter เฉพาะ `is_technical_valid = true`
- select เฉพาะ business columns หลัก
- ไม่ต้อง write table เพิ่มในวันนี้
- ใช้ `.count()` และ `.show()` เพื่อตรวจผล

Expected result:

- ได้เฉพาะ records ที่ผ่าน technical validity เบื้องต้น
- records ที่มี invalid amount/date/missing key/unknown status ยังอยู่ใน Silver table แต่ไม่ถูกเลือกเข้า `df_downstream_ready_orders`


In [16]:
df_downstream_ready_orders = (
    df_silver_from_table
    .filter(F.col("is_technical_valid") == True)
    .select(
        "order_id",
        "customer_id",
        "customer_name",
        "order_date",
        "order_amount",
        "payment_status",
        "source_system",
    )
    .orderBy("order_id")
)

df_downstream_ready_orders.show(truncate=False)
print("Downstream-ready record count:", df_downstream_ready_orders.count())

StatementMeta(, 81d83f9e-3ec9-4aae-b45c-eab6f27881ea, 18, Finished, Available, Finished, False)

+--------+-----------+-------------+----------+------------+--------------+-------------+
|order_id|customer_id|customer_name|order_date|order_amount|payment_status|source_system|
+--------+-----------+-------------+----------+------------+--------------+-------------+
|O1001   |C001       |Alice        |2026-05-01|1200.50     |success       |pos          |
|O1002   |C002       |Bob          |2026-05-01|900.00      |success       |pos          |
+--------+-----------+-------------+----------+------------+--------------+-------------+

Downstream-ready record count: 2


## Common mistakes

1. **แก้ Bronze table ทับ raw data**

   Bronze ควรรักษา raw/source traceability ไว้ ถ้าต้อง clean ควรสร้าง Silver output แยก ไม่ใช่ทำให้ Bronze เสียหลักฐานเดิม

2. **Cast ก่อน clean raw string**

   ถ้า amount มี comma, currency text หรือค่าว่าง ควร clean ก่อน cast ไม่อย่างนั้นอาจได้ null หรือ error จาก parsing

3. **ใช้ `dropDuplicates()` แบบไม่มี ordering rule**

   ถ้า duplicate records เป็นคนละ version กัน ต้องรู้ว่าจะเก็บ record ไหน เช่น latest `source_updated_at` ไม่ใช่ให้ Spark เลือกแบบไม่ชัดเจน

4. **Dedup ก่อนสร้าง issue flags จนปัญหาหายไป**

   ถ้า record ที่ถูก dedup ทิ้งมี data issue เราอาจมองไม่เห็นว่าต้นทางเคยส่งปัญหามา ควรตรวจ duplicate และ issue visibility ก่อนเสมอ

5. **Drop invalid records เงียบ ๆ**

   ถ้า invalid records ถูกลบทิ้งโดยไม่มี flag/log/quarantine จะ debug ยากมากเมื่อ business ถามว่าข้อมูลหายไปไหน

6. **ตัด audit columns ออกเร็วเกินไป**

   `source_file`, `batch_id`, `run_id`, `ingestion_timestamp` ยังมีประโยชน์มากใน Silver สำหรับ trace issue กลับไปหา Bronze/source

7. **เอา business aggregation มาปนใน Silver cleansing**

   Silver ควรเน้น standardized and cleaned records ส่วน aggregation เพื่อ reporting มักเหมาะกับ Gold มากกว่า


## Expected Output / Success Criteria

เมื่อจบ Day 20 ควรทำได้:

- อธิบายได้ว่า Bronze เน้น raw preservation/traceability ส่วน Silver เน้น standardized and cleaned data
- ทำ string standardization เช่น trim, lower, upper, normalize blank values ได้
- cast raw fields เป็น `DateType`, `TimestampType` และ `DecimalType` ได้
- normalize payment status เป็นกลุ่มมาตรฐาน เช่น `success`, `failed`, `pending`, `unknown` ได้
- สร้าง `technical_issue_reason` และ `is_technical_valid` เพื่อมองเห็น records ที่มี issue ได้
- ตรวจ duplicate candidates ก่อน dedup ได้
- deduplicate records ด้วย business key + latest timestamp rule ได้
- ตรวจผลหลัง write ด้วย `.show()`, `.printSchema()`, `.count()` และ duplicate key check ได้


## Final takeaway

Silver cleansing pattern คือขั้นที่ทำให้ข้อมูลจาก Bronze เริ่ม “ใช้งานได้จริง” มากขึ้น แต่ยังต้องไม่ทำให้ traceability หรือ issue visibility หายไป

> Good Silver data is not only clean. It is typed, standardized, deduplicated, and still traceable.

จำ mindset หลักของวันนี้ไว้:

- Bronze เก็บหลักฐานจาก source
- Silver ทำให้ข้อมูลเป็นระเบียบและ typed
- dedup ต้องมี business key และ ordering rule
- invalid records ไม่ควรหายเงียบ
- audit columns ยังช่วย debug ได้มากแม้อยู่ใน Silver layer


## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day20_bronze_orders_raw")
# spark.sql("DROP TABLE IF EXISTS day20_silver_orders")